# M2 · E4 Inferensi gambar penuh dan SAHI

Bandingkan satu checkpoint E2 native 1280 lima epoch pada dua alur inferensi. Kedua alur memproses 356 gambar validasi yang sama dengan 2.205 kotak acuan. Alur SAHI memakai tile 640 piksel, overlap 25%, dan NMS IoU 0,50. Evaluator mengembalikan semua kotak ke koordinat referensi 640.

Kaggle menjalankan inferensi ulang dari checkpoint historis yang fingerprint-nya diperiksa di bawah. Eksperimen ini menghasilkan prediksi, contoh visual, dan latensi CPU untuk kedua alur dari satu sesi. Data test tidak dipasang di notebook ini.

In [ ]:
import os
import subprocess
import sys

os.environ["YOLO_CONFIG_DIR"] = "/tmp/ciringoni-yolo"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.210", "pycocotools==2.0.11", "sahi==0.14.0"],
    check=True,
)

In [ ]:
import hashlib
import json
import tarfile
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

SOURCE_COMMIT = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
E2_SHA256 = "0bd467aef478ebd26555074030ff653c52cf05ad5380fcbe66057b3c995f264e"

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

scratch = TemporaryDirectory(prefix="ciringoni-e4-")
project = Path(scratch.name) / "Data_Mining_Ciringoni_M1"
project.mkdir()
input_root = Path("/kaggle/input")
archives = sorted(input_root.rglob("validation_e2.tar.gz"))
if archives:
    assert len(archives) == 1
    with tarfile.open(archives[0]) as archive:
        archive.extractall(project, filter="data")
    data = project
else:
    manifests = sorted(input_root.rglob("geometry.json"))
    assert len(manifests) == 1, "Lampirkan dataset privat validasi E2."
    data = manifests[0].parent

gt = data / "hasil_m1/coco_val_gt.json"
ground_truth = json.loads(gt.read_text())
assert (len(ground_truth["images"]), len(ground_truth["annotations"])) == (356, 2205)
checkpoint = data / "1280/best.pt"
assert sha256(checkpoint) == E2_SHA256
manifest = data / "geometry.json"
assert json.loads(manifest.read_text())["ready"]

gt_for_evaluator = project / "hasil_m1/coco_val_gt.json"
gt_for_evaluator.parent.mkdir(exist_ok=True)
if gt != gt_for_evaluator:
    gt_for_evaluator.write_bytes(gt.read_bytes())
sources = [
    "hasil_m2/evaluasi_e4_sahi.py", "hasil_m2/evaluasi_e2.py",
    "hasil_m2/evaluasi_mosaic.py", "hasil_m2/uji_tiling_640.py",
]
source_files = {}
for name in sources:
    destination = project / name
    destination.parent.mkdir(exist_ok=True)
    urlretrieve(
        f"https://raw.githubusercontent.com/Jordyee/ciringoni-data-mining/{SOURCE_COMMIT}/Data_Mining_Ciringoni_M1/{name}",
        destination,
    )
    source_files[name] = sha256(destination)
print("Validation: 356 images, 2205 boxes. E2 checkpoint SHA-256:", E2_SHA256)
print("Source commit:", SOURCE_COMMIT)

In [ ]:
from importlib.metadata import version
from IPython.display import Code, display

output = Path("/kaggle/working/E4")
evaluator = project / "hasil_m2/evaluasi_e4_sahi.py"
display(Code(filename=str(evaluator), language="python"))
command = [
    sys.executable, str(evaluator), str(checkpoint),
    str(data / "1280/images/val"), str(manifest), str(output),
    "--device", "cpu", "--evidence-reference", str(data / "640/images/val"),
]
subprocess.run([*command, "--smoke"], check=True)
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Image
import pandas as pd

metrics = json.loads((output / "metrik_e4_sahi.json").read_text())
assert metrics["checkpoint_sha256"] == E2_SHA256
assert (metrics["images"], metrics["gt_boxes"], metrics["slices_per_image"]) == (356, 2205, 9)
assert metrics["torch_cpu_threads"] == 4
evidence_hashes = {}
rows = {}
for mode in ("full", "sahi"):
    predictions = output / f"prediksi_{mode}.json"
    predictions_list = json.loads(predictions.read_text())
    assert predictions_list
    assert {row["image_id"] for row in predictions_list} <= set(range(1, 357))
    assert 0 < metrics[mode]["cpu_median_ms"] <= metrics[mode]["cpu_p95_ms"]
    evidence_hashes[predictions.name] = sha256(predictions)
    rows[mode] = metrics[mode] | {"predictions": len(predictions_list)}
display(pd.DataFrame.from_dict(rows, orient="index"))
display(Image(filename=str(output / "bukti_e4_sahi.png")))
for name in ("metrik_e4_sahi.json", "bukti_e4_sahi.png"):
    evidence_hashes[name] = sha256(output / name)
proof = {
    "experiment": "E4",
    "activity": "Kaggle CPU inference, COCO validation and CPU benchmark",
    "checkpoint_origin": "historical E2 native 1280, five epochs",
    "checkpoint_sha256": E2_SHA256,
    "source_commit": SOURCE_COMMIT,
    "source_sha256": source_files,
    "dataset": "davejordygerungan/ciringoni-m2-validasi-e2",
    "geometry_sha256": sha256(manifest), "ground_truth_sha256": sha256(gt),
    "version": {p: version(p) for p in ("ultralytics", "pycocotools", "sahi", "torch", "torchvision")},
    "python": sys.version.split()[0],
    "cpu": next(line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name")),
    "images": metrics["images"], "gt_boxes": metrics["gt_boxes"],
    "predictions": {mode: row["predictions"] for mode, row in rows.items()},
    "output_sha256": evidence_hashes,
}
(output / "run_e4.json").write_text(json.dumps(proof, indent=2) + "\n")
print("Verified E4 outputs:", sorted(evidence_hashes))